Test Script for MSConvNeXt Grid Search Models

Matches train_MSconvnext_grid_search.ipynb with visualization features:
✓ Same model architecture (Multi-Scale Fusion)
✓ GradCAM visualization
✓ Saliency maps
✓ Attention maps
✓ Support for grid search trial models
✓ Single image prediction with visualizations


In [ ]:
# =============================================================================
# Install Dependencies (Run this cell first if you get ModuleNotFoundError)
# =============================================================================
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])
        return True
    except:
        return False

# Check and install required packages
required_packages = {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'PIL': 'Pillow',
    'cv2': 'opencv-python',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn'
}

missing_packages = []
for module_name, package_name in required_packages.items():
    try:
        __import__(module_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print("Installing missing packages...")
    print(f"Missing: {', '.join(missing_packages)}")
    for package in missing_packages:
        print(f"Installing {package}...")
        if install_package(package):
            print(f"✓ {package} installed successfully")
        else:
            print(f"✗ Failed to install {package}. Please install manually: pip install {package}")
    print("\nPlease restart the kernel after installation and run this cell again.")
else:
    print("✓ All required packages are installed!")

# Now import the packages
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import json
import glob

print(f"\n✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")


Installing missing packages...
Missing: torch, torchvision, opencv-python, seaborn, scikit-learn
Installing torch...


In [ ]:
# =============================================================================
# Configuration (Matching Grid Search Training Script)
# =============================================================================
DATASET_DIR = '../../dataset'
SAVE_DIR = '../../saved_models_and_data'
SPLIT_OUTPUT_DIR = '../../dataset_split'
TEST_IMAGES_DIR = '../test_images'

IMAGE_SIZE = (320, 320)

# Model paths
MODEL_FILENAME = 'wheat_disease_convnext_model.pth'
MODEL_PATH = os.path.join(SAVE_DIR, MODEL_FILENAME)
LEGACY_MODEL_PATH = os.path.join(SAVE_DIR, 'best_model_simple.pth')
GRID_SEARCH_DIR = os.path.join(SAVE_DIR, 'grid_search_msconvnext')

# Test configuration
USE_TTA = True  # Test-time augmentation
TTA_N_AUGMENTS = 7


In [ ]:
# =============================================================================
# Class Labels (Dynamic Loading)
# =============================================================================
def get_class_labels(dataset_dir):
    return sorted([d for d in os.listdir(dataset_dir) 
                   if os.path.isdir(os.path.join(dataset_dir, d))])

class_labels = get_class_labels(DATASET_DIR)
print(f"Classes: {class_labels}")
print(f"Number of classes: {len(class_labels)}")


In [ ]:
# =============================================================================
# Test Transforms (Same as Training Script)
# =============================================================================
test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
# =============================================================================
# Multi-Scale Fusion Module (Same as Training Script)
# =============================================================================
class MultiScaleFusion(nn.Module):
    """
    Multi-scale feature fusion with 3 branches
    This is our main contribution - captures disease features at different scales
    """
    def __init__(self, channels):
        super().__init__()
        
        # Three branches: 3x3, 5x5, 7x7 convolutions
        self.branch1 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(channels, channels, 5, padding=2, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(channels, channels, 7, padding=3, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        # Fuse all branches
        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 1),
            nn.BatchNorm2d(channels)
        )
    
    def forward(self, x):
        f1 = self.branch1(x)  # Fine details
        f2 = self.branch2(x)  # Medium patterns
        f3 = self.branch3(x)  # Large context
        
        # Concatenate and fuse
        concat = torch.cat([f1, f2, f3], dim=1)
        fused = self.fusion(concat)
        
        return fused


In [ ]:
# =============================================================================
# Build Model (Same as Training Script)
# =============================================================================
def build_model(num_classes):
    """Build ConvNeXt with Multi-Scale Fusion"""
    
    # Load pretrained ConvNeXt (matching training script)
    model = models.convnext_base(pretrained=True)
    in_features = model.classifier[2].in_features
    
    # Add our multi-scale fusion module
    model.fusion = MultiScaleFusion(in_features)
    
    # Enhanced classifier head for better performance
    model.classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.LayerNorm(in_features),
        nn.Dropout(0.2),  # Added early dropout
        nn.Linear(in_features, 768),  # Increased from 512
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(768, 384),  # Added intermediate layer
        nn.GELU(),
        nn.Dropout(0.2),
        nn.Linear(384, num_classes)
    )
    
    # Custom forward pass
    def forward(x):
        x = model.features(x)     # ConvNeXt backbone
        x = model.fusion(x)       # Our multi-scale fusion
        x = model.classifier(x)   # Classification head
        return x
    
    model.forward = forward
    return model


In [ ]:
# =============================================================================
# Test-Time Augmentation (Same as Training Script)
# =============================================================================
def test_time_augmentation(model, inputs, device, n_augments=7):
    """Apply TTA for more robust predictions - averages predictions from augmented versions"""
    model.eval()
    predictions = []
    
    # Original prediction
    with torch.no_grad():
        outputs = model(inputs)
        predictions.append(F.softmax(outputs, dim=1))
    
    # Augmented versions
    for _ in range(n_augments - 1):
        aug_inputs = inputs.clone()
        
        # Random augmentation
        aug_type = np.random.randint(3)
        
        if aug_type == 0:  # Horizontal flip
            aug_inputs = torch.flip(aug_inputs, [3])
        elif aug_type == 1:  # Vertical flip
            aug_inputs = torch.flip(aug_inputs, [2])
        elif aug_type == 2:  # Color jitter (simplified)
            brightness = 0.9 + 0.2 * np.random.rand()
            aug_inputs = aug_inputs * brightness
            aug_inputs = torch.clamp(aug_inputs, 0, 1)
        
        with torch.no_grad():
            outputs = model(aug_inputs)
            predictions.append(F.softmax(outputs, dim=1))
    
    # Average predictions
    avg_pred = torch.stack(predictions).mean(0)
    return avg_pred


In [ ]:
# =============================================================================
# Load Model Function
# =============================================================================
def load_model(num_classes, device, model_path=None, trial_name=None):
    """Load trained model - supports grid search trials"""
    
    # Determine checkpoint path
    if model_path:
        checkpoint_path = model_path
    elif trial_name:
        # Load from grid search trial
        checkpoint_path = os.path.join(
            SAVE_DIR, f"best_msconvnext_model_{trial_name.replace(' ', '_')}.pth"
        )
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"Trial model not found: {checkpoint_path}")
    else:
        # Default: try standard paths
        checkpoint_path = MODEL_PATH if os.path.exists(MODEL_PATH) else LEGACY_MODEL_PATH
        if checkpoint_path == LEGACY_MODEL_PATH and not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"No checkpoint found at {MODEL_PATH} or {LEGACY_MODEL_PATH}")
    
    if checkpoint_path == LEGACY_MODEL_PATH:
        print(f"⚠️ Using legacy checkpoint: {LEGACY_MODEL_PATH}")
    elif trial_name:
        print(f"✓ Loading grid search trial: {trial_name}")
        print(f"  Path: {checkpoint_path}")
    else:
        print(f"✓ Loading model from: {checkpoint_path}")
    
    # Build model
    model = build_model(num_classes).to(device)
    
    # Load weights
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✓ Model loaded with {num_params:.1f}M parameters\n")
    
    return model


In [ ]:
# =============================================================================
# Visualization Functions (GradCAM, Saliency, Attention)
# =============================================================================
class GradCAM:
    """Improved Grad-CAM for accurate disease spot highlighting"""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        
        def backward_hook(module, grad_in, grad_out):
            # Handle cases where grad_out might be None or tuple
            if grad_out[0] is not None:
                self.gradients = grad_out[0].detach()
            else:
                self.gradients = None
        
        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))
    
    def __call__(self, input_tensor, class_idx=None, threshold_percentile=75):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        loss = output[0, class_idx]
        loss.backward()
        
        if self.gradients is None:
            # Fallback: create zero gradients
            self.gradients = torch.zeros_like(self.activations)
        
        gradients = self.gradients[0]
        activations = self.activations[0]
        
        # Improved weight computation: use global average pooling with ReLU
        weights = torch.relu(gradients).mean(dim=(1, 2))
        
        # Weighted combination of activations
        cam = (weights[:, None, None] * activations).sum(dim=0)
        cam = torch.relu(cam)
        cam = cam.cpu().numpy()
        
        # Resize to original image size
        cam = cv2.resize(cam, IMAGE_SIZE, interpolation=cv2.INTER_CUBIC)
        
        # Improved normalization with thresholding for better focus
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max > cam_min:
            cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
            # Apply threshold to focus on high-activation regions
            threshold = np.percentile(cam, threshold_percentile)
            cam = np.clip((cam - threshold) / (1 - threshold + 1e-8), 0, 1)
        else:
            cam = np.zeros_like(cam)
        
        return cam
    
    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()


def compute_saliency_map(model, input_tensor, class_idx=None, threshold_percentile=70):
    """Improved saliency map computation"""
    input_tensor = input_tensor.clone().detach().requires_grad_(True)
    model.zero_grad()
    output = model(input_tensor)
    if class_idx is None:
        class_idx = output.argmax(dim=1).item()
    
    loss = output[0, class_idx]
    loss.backward()
    
    # Get gradients and compute saliency
    saliency = input_tensor.grad.data.abs().squeeze().cpu().numpy()
    saliency = np.max(saliency, axis=0)
    
    # Resize with better interpolation
    saliency = cv2.resize(saliency, IMAGE_SIZE, interpolation=cv2.INTER_CUBIC)
    
    # Normalize with thresholding
    sal_min, sal_max = saliency.min(), saliency.max()
    if sal_max > sal_min:
        saliency = (saliency - sal_min) / (sal_max - sal_min + 1e-8)
        # Apply threshold to focus on important regions
        threshold = np.percentile(saliency, threshold_percentile)
        saliency = np.clip((saliency - threshold) / (1 - threshold + 1e-8), 0, 1)
    else:
        saliency = np.zeros_like(saliency)
    
    return saliency


def compute_attention_map(model, input_tensor):
    """Feature attention map from fusion layer"""
    with torch.no_grad():
        features = None
        def hook_fn(module, input, output):
            nonlocal features
            features = output.detach()
        
        # Use fusion layer for better disease-specific features
        handle = model.fusion.register_forward_hook(hook_fn)
        _ = model(input_tensor)
        handle.remove()
        
        if features is not None:
            attn_map = features.mean(dim=1).squeeze().cpu().numpy()
            attn_map = cv2.resize(attn_map, IMAGE_SIZE, interpolation=cv2.INTER_CUBIC)
            attn_min, attn_max = attn_map.min(), attn_map.max()
            if attn_max > attn_min:
                attn_map = (attn_map - attn_min) / (attn_max - attn_min + 1e-8)
            else:
                attn_map = np.zeros_like(attn_map)
        else:
            attn_map = np.zeros(IMAGE_SIZE)
        
        return attn_map


def show_cam_on_image(img: np.ndarray, mask: np.ndarray, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """Improved overlay with better blending for disease visualization"""
    # Use HOT colormap for disease (red/yellow) or JET for general
    heatmap = cv2.applyColorMap(np.uint8(255 * mask), colormap)
    heatmap = np.float32(heatmap) / 255.0
    
    # Better blending: emphasize disease regions
    # Use alpha blending with enhanced contrast for disease spots
    enhanced_mask = np.power(mask, 0.7)  # Gamma correction for better visibility
    heatmap_weighted = heatmap * enhanced_mask[:, :, np.newaxis]
    
    # Blend with original image
    img_float = np.float32(img)
    cam = (1 - alpha) * img_float + alpha * heatmap_weighted
    cam = np.clip(cam, 0, 1)
    
    return np.uint8(255 * cam)


# Load Model

Choose one of the following:
1. Default model (standard training)
2. Grid search trial model (specify trial_name)
3. Custom model path


In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Load model - modify these parameters as needed
USE_GRID_SEARCH_TRIAL = False  # Set to True to use grid search trial
TRIAL_NAME = None  # e.g., "trial1_lr0.0001_bs24"
CUSTOM_MODEL_PATH = None  # e.g., "../../saved_models_and_data/custom_model.pth"

# Load model
model = load_model(
    num_classes=len(class_labels),
    device=device,
    model_path=CUSTOM_MODEL_PATH,
    trial_name=TRIAL_NAME if USE_GRID_SEARCH_TRIAL else None
)

# Initialize GradCAM with fusion layer
target_layer = model.fusion
gradcam = GradCAM(model, target_layer)


# List Available Grid Search Trials (Optional)


In [ ]:
# List available grid search trials
if os.path.exists(GRID_SEARCH_DIR):
    trial_files = glob.glob(os.path.join(SAVE_DIR, "best_msconvnext_model_*.pth"))
    if trial_files:
        print("Available Grid Search Trials:")
        print("="*80)
        for trial_file in sorted(trial_files):
            trial_name = os.path.basename(trial_file).replace("best_msconvnext_model_", "").replace(".pth", "")
            print(f"  {trial_name}")
        print("="*80)
        
        # Try to load leaderboard
        leaderboard_path = os.path.join(GRID_SEARCH_DIR, "leaderboard.json")
        if os.path.exists(leaderboard_path):
            with open(leaderboard_path, 'r') as f:
                leaderboard = json.load(f)
            print("\nTop 5 Trials (by validation accuracy):")
            for i, trial in enumerate(leaderboard[:5], 1):
                print(f"  {i}. {trial['trial_name']} | "
                      f"lr={trial['learning_rate']} | bs={trial['batch_size']} | "
                      f"val_acc={trial['best_val_acc']:.4f} | "
                      f"test_acc={trial['test_acc']:.4f}")
else:
    print("No grid search directory found.")


# Single Image Prediction with Visualizations


In [ ]:
# Process all images in test directory
if not os.path.exists(TEST_IMAGES_DIR):
    print(f"Test images directory not found: {TEST_IMAGES_DIR}")
    print("Please create the directory and add test images.")
else:
    image_files = [f for f in os.listdir(TEST_IMAGES_DIR) 
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    
    if not image_files:
        print(f"No images found in {TEST_IMAGES_DIR}")
    else:
        print(f"Found {len(image_files)} images to process\n")
        
        for img_name in image_files:
            img_path = os.path.join(TEST_IMAGES_DIR, img_name)
            img_pil = Image.open(img_path).convert('RGB')
            img_tensor = test_transform(img_pil).unsqueeze(0).to(device)
            
            # Get prediction (with TTA if enabled)
            with torch.no_grad():
                if USE_TTA:
                    outputs = test_time_augmentation(model, img_tensor, device, n_augments=TTA_N_AUGMENTS)
                else:
                    outputs = model(img_tensor)
                    outputs = F.softmax(outputs, dim=1)
                
                pred_idx = outputs.argmax(dim=1).item()
                pred_label = class_labels[pred_idx]
                prob = outputs[0, pred_idx].item()
            
            img_np = np.array(img_pil.resize(IMAGE_SIZE)).astype(np.float32) / 255.0
            
            # Skip Grad-CAM and heatmaps for "healthy" class
            is_healthy = pred_label.lower() == 'healthy'
            
            if is_healthy:
                # For healthy images, just show original and prediction info
                all_probs = outputs[0].cpu().numpy()
                plt.figure(figsize=(12, 5))
                plt.subplot(1, 2, 1)
                plt.imshow(img_pil.resize(IMAGE_SIZE))
                plt.title(f'Original Image\n{img_name}', fontsize=12, fontweight='bold')
                plt.axis('off')
                
                plt.subplot(1, 2, 2)
                # Show top predictions
                top_k = min(5, len(class_labels))
                top_indices = np.argsort(all_probs)[-top_k:][::-1]
                top_labels = [class_labels[i] for i in top_indices]
                top_probs = [all_probs[i] for i in top_indices]
                
                colors = ['green' if label == pred_label else 'gray' for label in top_labels]
                bars = plt.barh(range(len(top_labels)), top_probs, color=colors)
                plt.yticks(range(len(top_labels)), top_labels)
                plt.xlabel('Probability', fontsize=11)
                plt.title(f'Prediction: {pred_label.upper()}\nConfidence: {prob:.1%}', 
                         fontsize=12, fontweight='bold')
                plt.xlim(0, 1)
                plt.gca().invert_yaxis()
                plt.grid(axis='x', alpha=0.3)
                
                plt.suptitle(f'✓ HEALTHY - No Disease Detected', 
                            fontsize=14, fontweight='bold', color='green')
                plt.tight_layout()
                plt.show()
            else:
                # For disease images, compute all heatmaps
                # Use JET colormap for true Grad-CAM visualization (blue to red spectrum)
                cam = gradcam(img_tensor, class_idx=pred_idx, threshold_percentile=75)
                saliency = compute_saliency_map(model, img_tensor, class_idx=pred_idx, threshold_percentile=70)
                attn_map = compute_attention_map(model, img_tensor)
                
                # Create overlays with JET colormap for true Grad-CAM (traditional blue to red)
                cam_img = show_cam_on_image(img_np, cam, alpha=0.45, colormap=cv2.COLORMAP_JET)
                sal_img = show_cam_on_image(img_np, saliency, alpha=0.45, colormap=cv2.COLORMAP_JET)
                attn_img = show_cam_on_image(img_np, attn_map, alpha=0.4, colormap=cv2.COLORMAP_JET)
                
                # Create visualization
                fig = plt.figure(figsize=(22, 5))
                
                # Original image
                plt.subplot(1, 6, 1)
                plt.imshow(img_pil.resize(IMAGE_SIZE))
                plt.title('Original Image', fontsize=11, fontweight='bold')
                plt.axis('off')
                
                # Grad-CAM heatmap (JET colormap - true Grad-CAM)
                plt.subplot(1, 6, 2)
                plt.imshow(cam, cmap='jet', vmin=0, vmax=1)
                plt.title('Grad-CAM\n(Disease Regions)', fontsize=11, fontweight='bold')
                plt.axis('off')
                plt.colorbar(fraction=0.046, pad=0.04)
                
                # Grad-CAM overlay
                plt.subplot(1, 6, 3)
                plt.imshow(cam_img)
                plt.title('Grad-CAM Overlay\n(Highlighted)', fontsize=11, fontweight='bold')
                plt.axis('off')
                
                # Saliency overlay
                plt.subplot(1, 6, 4)
                plt.imshow(sal_img)
                plt.title('Saliency Map\n(Important Pixels)', fontsize=11, fontweight='bold')
                plt.axis('off')
                
                # Attention overlay
                plt.subplot(1, 6, 5)
                plt.imshow(attn_img)
                plt.title('Feature Attention\n(Fusion Layer)', fontsize=11, fontweight='bold')
                plt.axis('off')
                
                # Prediction info
                plt.subplot(1, 6, 6)
                # Show top predictions
                all_probs = outputs[0].cpu().numpy()
                top_k = min(5, len(class_labels))
                top_indices = np.argsort(all_probs)[-top_k:][::-1]
                top_labels = [class_labels[i] for i in top_indices]
                top_probs = [all_probs[i] for i in top_indices]
                
                colors = ['red' if label == pred_label else 'gray' for label in top_labels]
                bars = plt.barh(range(len(top_labels)), top_probs, color=colors)
                plt.yticks(range(len(top_labels)), top_labels)
                plt.xlabel('Probability', fontsize=10)
                plt.title(f'Top Predictions', fontsize=11, fontweight='bold')
                plt.xlim(0, 1)
                plt.gca().invert_yaxis()
                plt.grid(axis='x', alpha=0.3)
                
                # Main title
                tta_text = " (with TTA)" if USE_TTA else ""
                plt.suptitle(f'⚠️ DISEASE DETECTED: {pred_label.upper().replace("_", " ")} | Confidence: {prob:.1%}{tta_text} | Image: {img_name}', 
                            fontsize=13, fontweight='bold', color='red')
                plt.tight_layout()
                plt.show()
            
            # Clean up hooks after each image to prevent memory issues
            gradcam.remove_hooks()
            gradcam = GradCAM(model, target_layer)
